# Notebook 2 — Merge USA Medal Days with Market Data + Abnormal Returns (CAPM / Fama–French)

This notebook reads the medal-day panel (with `gold/silver/bronze/performance_score`) and builds a daily panel that includes:

- **SPY** market return, volume change, and volatility proxies
- **Sector / sentiment-exposed tickers** (default: `NKE`, `XLY`, `DKNG`) returns + volume + volatility
- **Abnormal returns** for the tickers using:
  - **CAPM** (Market factor only) and/or
  - **Fama–French 3-factor** (MKT-RF, SMB, HML)

It outputs a single CSV panel that can be used for regressions/event-study.

> Notes:
> - This notebook uses **online data sources** at runtime (Yahoo Finance + Ken French/FamaFrench via `pandas_datareader`). If your environment blocks web access, you can download the data manually and point the notebook to local CSVs.
> - Medal dates that fall on non-trading days are **shifted to the next NYSE trading day**.


In [15]:
# Install dependencies if needed (comment out if already installed)
%pip -q install pandas numpy yfinance pandas_datareader pandas_market_calendars statsmodels tqdm

Note: you may need to restart the kernel to use updated packages.


In [16]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
from datetime import timedelta

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import yfinance as yf
import pandas_market_calendars as mcal
from pandas_datareader import data as pdr
import statsmodels.api as sm
from statsmodels.regression.rolling import RollingOLS

pd.set_option("display.max_columns", 200)

## 1) Load the medal-day dataset

- `usa_medal_daily_counts_and_score.csv` or similar

Expected columns (minimum):
- `event_date_end`
- `gold`, `silver`, `bronze`
- `performance_score`
- `total_medals`

If your file has `date` instead of `event_date_end`, the notebook will handle it.


In [17]:
MEDAL_PATH = "../data/interim/usa/usa_medal_daily_counts_and_score_ALL.csv"
medals = pd.read_csv(MEDAL_PATH)

print("Shape:", medals.shape)
display(medals.head())
print("Columns:", list(medals.columns))

Shape: (281, 7)


,event_date_end,total_medals,gold,silver,bronze,performance_score,season
0,2000-01-01,23,10,6,7,49,S
1,2000-02-01,13,3,5,5,24,S
2,2000-02-16,2,0,0,2,2,S
3,2000-03-01,1,0,0,1,1,S
4,2000-04-01,2,2,0,0,6,S


Columns: ['event_date_end', 'total_medals', 'gold', 'silver', 'bronze', 'performance_score', 'season']


In [18]:
# Normalize date column name
if "event_date_end" in medals.columns:
    medals = medals.rename(columns={"event_date_end": "medal_date"})
elif "Date" in medals.columns:
    medals = medals.rename(columns={"Date": "medal_date"})
elif "date" in medals.columns:
    medals = medals.rename(columns={"date": "medal_date"})
else:
    raise ValueError("Could not find a date column. Expected 'event_date_end' or 'date'.")

medals["medal_date"] = pd.to_datetime(medals["medal_date"], errors="coerce")
medals = medals.dropna(subset=["medal_date"]).sort_values("medal_date").reset_index(drop=True)

# Ensure standard columns exist
for c in ["gold","silver","bronze"]:
    if c not in medals.columns:
        raise ValueError(f"Missing required column '{c}' in medal-day CSV.")

if "total_medals" not in medals.columns:
    medals["total_medals"] = medals[["gold","silver","bronze"]].sum(axis=1)

if "performance_score" not in medals.columns:
    medals["performance_score"] = 3*medals["gold"] + 2*medals["silver"] + 1*medals["bronze"]

if "any_medal_dummy" not in medals.columns:
    medals["any_medal_dummy"] = (medals["total_medals"] > 0).astype(int)

display(medals.head())
print("Medal date range:", medals["medal_date"].min().date(), "→", medals["medal_date"].max().date())

,medal_date,total_medals,gold,silver,bronze,performance_score,season,any_medal_dummy
0,2000-01-01,23,10,6,7,49,S,1
1,2000-02-01,13,3,5,5,24,S,1
2,2000-02-16,2,0,0,2,2,S,1
3,2000-03-01,1,0,0,1,1,S,1
4,2000-04-01,2,2,0,0,6,S,1


Medal date range: 2000-01-01 → 2022-11-25


## 2) Align medal dates to NYSE trading days

Olympic events happen on weekends and holidays. For market merges, we map each medal date to the **next NYSE trading day** (if not already a trading day).

This yields: `trade_date`.


In [19]:
nyse = mcal.get_calendar("NYSE")

start = medals["medal_date"].min() - pd.Timedelta(days=10)
end = medals["medal_date"].max() + pd.Timedelta(days=10)

schedule = nyse.schedule(start_date=start, end_date=end)
trading_days = pd.DatetimeIndex(schedule.index)

def to_next_trading_day(d: pd.Timestamp) -> pd.Timestamp:
    if d in trading_days:
        return d
    idx = trading_days.searchsorted(d)
    if idx >= len(trading_days):
        return pd.NaT
    return trading_days[idx]

medals["trade_date"] = medals["medal_date"].apply(to_next_trading_day)
medals = medals.dropna(subset=["trade_date"]).sort_values("trade_date").reset_index(drop=True)

display(medals.head(10)[["medal_date","trade_date","gold","silver","bronze","performance_score"]])
print("Unique trading days:", medals["trade_date"].nunique())

,medal_date,trade_date,gold,silver,bronze,performance_score
0,2000-01-01,2000-01-03,10,6,7,49
1,2000-02-01,2000-02-01,3,5,5,24
2,2000-02-16,2000-02-16,0,0,2,2
3,2000-03-01,2000-03-01,0,0,1,1
4,2000-04-01,2000-04-03,2,0,0,6
5,2000-04-23,2000-04-24,0,0,1,1
6,2000-05-10,2000-05-10,0,0,1,1
7,2000-06-01,2000-06-01,1,0,1,4
8,2000-07-01,2000-07-03,1,0,0,3
9,2000-07-09,2000-07-10,1,0,0,3


Unique trading days: 256


## 3) Download market data (Yahoo Finance)

Default tickers:
- `SPY` (market proxy)
- `NKE` (Nike)
- `XLY` (Consumer Discretionary ETF)
- `DKNG` (DraftKings)

You can edit `TICKERS`.


In [20]:
TICKERS = ["SPY", "NKE", "XLY", "DKNG"] 

fetch_start = (medals["trade_date"].min() - pd.Timedelta(days=365)).date()
fetch_end = (medals["trade_date"].max() + pd.Timedelta(days=10)).date()

print("Fetching price data from", fetch_start, "to", fetch_end)
raw = yf.download(TICKERS, start=str(fetch_start), end=str(fetch_end), auto_adjust=False, progress=False)

# Extract fields
prices = raw["Adj Close"] if "Adj Close" in raw else raw["Close"]
close = raw["Close"]
volume = raw["Volume"]

# Ensure DataFrames (yfinance returns Series when one ticker)
if isinstance(prices, pd.Series): prices = prices.to_frame()
if isinstance(close, pd.Series): close = close.to_frame()
if isinstance(volume, pd.Series): volume = volume.to_frame()

display(prices.head())
display(volume.head())

Fetching price data from 1999-01-03 to 2022-12-05


Ticker,DKNG,NKE,SPY,XLY
Date,,,,
1999-01-04,NaN,3.795359,76.677505,9.582786
1999-01-05,NaN,3.737942,77.553932,9.744226
1999-01-06,NaN,4.007808,79.423622,10.020988
1999-01-07,NaN,3.944645,79.034157,10.009456
1999-01-08,NaN,4.030773,79.618385,10.020988


Ticker,DKNG,NKE,SPY,XLY
Date,,,,
1999-01-04,NaN,9884800,9450400,10400
1999-01-05,NaN,5804000,8031000,2600
1999-01-06,NaN,28024000,7737700,16200
1999-01-07,NaN,12841600,5504900,27400
1999-01-08,NaN,9901600,6224400,37200


In [21]:
# Daily log returns
ret = np.log(prices).diff()

# Volume log change
log_volume_change = np.log(volume.replace(0, np.nan)).diff()

# Volatility proxies
abs_ret = ret.abs()              # close-to-close absolute return
vol_20d = ret.rolling(20).std()  # 20d rolling volatility

def to_long(df: pd.DataFrame, value_name: str) -> pd.DataFrame:
    out = df.stack().rename(value_name).reset_index()
    out.columns = ["date", "ticker", value_name]
    return out

market_long = (
    to_long(ret, "ret")
    .merge(to_long(log_volume_change, "log_volume_change"), on=["date","ticker"], how="left")
    .merge(to_long(abs_ret, "abs_ret"), on=["date","ticker"], how="left")
    .merge(to_long(vol_20d, "vol_20d"), on=["date","ticker"], how="left")
)

display(market_long.head(10))

,date,ticker,ret,log_volume_change,abs_ret,vol_20d
0,1999-01-05,NKE,-0.015244,-0.532451,0.015244,NaN
1,1999-01-05,SPY,0.011365,-0.162748,0.011365,NaN
2,1999-01-05,XLY,0.016707,-1.386294,0.016707,NaN
3,1999-01-06,NKE,0.069709,1.574514,0.069709,NaN
4,1999-01-06,SPY,0.023822,-0.037205,0.023822,NaN
5,1999-01-06,XLY,0.028007,1.829500,0.028007,NaN
6,1999-01-07,NKE,-0.015885,-0.780371,0.015885,NaN
7,1999-01-07,SPY,-0.004916,-0.340466,0.004916,NaN
8,1999-01-07,XLY,-0.001152,0.525532,0.001152,NaN
9,1999-01-08,NKE,0.021599,-0.259994,0.021599,NaN


## 4) Load daily Fama–French factors

We pull **daily** factors and RF:
- MKT-RF, SMB, HML, RF (all in decimals)


In [22]:
ff = pdr.DataReader("F-F_Research_Data_Factors_daily", "famafrench")[0].copy()
ff.index = pd.to_datetime(ff.index)
ff = ff.rename(columns={"Mkt-RF":"MKT_RF"})
ff = ff[["MKT_RF","SMB","HML","RF"]].sort_index() / 100.0 

display(ff.head())
print("Factor date range:", ff.index.min().date(), "→", ff.index.max().date())

,MKT_RF,SMB,HML,RF
Date,,,,
2021-03-18,-0.0186,-0.0174,0.0216,0.0
2021-03-19,0.0023,0.0126,-0.0099,0.0
2021-03-22,0.0056,-0.0119,-0.0223,0.0
2021-03-23,-0.0116,-0.0270,-0.0104,0.0
2021-03-24,-0.0097,-0.0197,0.0146,0.0


Factor date range: 2021-03-18 → 2026-01-30


## 5) Abnormal returns (rolling CAPM + rolling FF3)

We compute abnormal returns as rolling-model residuals for each ticker **except SPY** (SPY is the market proxy).

Rolling window default: 252 trading days.


In [23]:
ROLLING_WINDOW = 252  # change to 120 if you prefer

def compute_abnormal_returns_for_ticker(ticker: str, ret_df: pd.DataFrame, ff: pd.DataFrame, window: int = 252) -> pd.DataFrame:
    r = ret_df[ticker].dropna()
    dfm = pd.DataFrame({"ret": r}).join(ff, how="inner").dropna()
    dfm["excess_ret"] = dfm["ret"] - dfm["RF"]

    # CAPM
    X_capm = sm.add_constant(dfm[["MKT_RF"]])
    y = dfm["excess_ret"]
    capm_fit = RollingOLS(y, X_capm, window=window, min_nobs=max(60, int(window*0.5))).fit()
    capm_pred = (X_capm * capm_fit.params).sum(axis=1)
    dfm["abret_capm"] = y - capm_pred

    # FF3
    X_ff3 = sm.add_constant(dfm[["MKT_RF","SMB","HML"]])
    ff3_fit = RollingOLS(y, X_ff3, window=window, min_nobs=max(60, int(window*0.5))).fit()
    ff3_pred = (X_ff3 * ff3_fit.params).sum(axis=1)
    dfm["abret_ff3"] = y - ff3_pred

    out = dfm[["abret_capm","abret_ff3"]].copy()
    out["date"] = out.index
    out["ticker"] = ticker
    return out.reset_index(drop=True)

ab_list = []
for t in [x for x in TICKERS if x != "SPY"]:
    ab_list.append(compute_abnormal_returns_for_ticker(t, ret, ff, window=ROLLING_WINDOW))

abrets_long = pd.concat(ab_list, ignore_index=True)
display(abrets_long.head())
print(abrets_long.groupby("ticker").size())

,abret_capm,abret_ff3,date,ticker
0,-0.011459,-0.011459,2021-03-18,NKE
1,-0.040481,-0.040481,2021-03-19,NKE
2,0.005657,0.005657,2021-03-22,NKE
3,-0.008352,-0.008352,2021-03-23,NKE
4,-0.029305,-0.029305,2021-03-24,NKE


ticker
DKNG    433
NKE     433
XLY     433
dtype: int64


In [24]:
# Merge abnormal returns into market_long
market_long = market_long.merge(abrets_long, on=["date","ticker"], how="left")
display(market_long.head(10))

,date,ticker,ret,log_volume_change,abs_ret,vol_20d,abret_capm,abret_ff3
0,1999-01-05,NKE,-0.015244,-0.532451,0.015244,NaN,NaN,NaN
1,1999-01-05,SPY,0.011365,-0.162748,0.011365,NaN,NaN,NaN
2,1999-01-05,XLY,0.016707,-1.386294,0.016707,NaN,NaN,NaN
3,1999-01-06,NKE,0.069709,1.574514,0.069709,NaN,NaN,NaN
4,1999-01-06,SPY,0.023822,-0.037205,0.023822,NaN,NaN,NaN
5,1999-01-06,XLY,0.028007,1.829500,0.028007,NaN,NaN,NaN
6,1999-01-07,NKE,-0.015885,-0.780371,0.015885,NaN,NaN,NaN
7,1999-01-07,SPY,-0.004916,-0.340466,0.004916,NaN,NaN,NaN
8,1999-01-07,XLY,-0.001152,0.525532,0.001152,NaN,NaN,NaN
9,1999-01-08,NKE,0.021599,-0.259994,0.021599,NaN,NaN,NaN


## 6) Build a wide market table for medal trading days and merge

We pivot ticker metrics into columns like:
- `ret_SPY`, `log_volume_change_SPY`, `vol_20d_SPY`
- `ret_NKE`, `abret_ff3_NKE`, etc.

Then merge onto medal-day panel by `trade_date`.


In [25]:
# Restrict market_long to medal trading days
trade_dates = pd.DatetimeIndex(sorted(medals["trade_date"].unique()))
mkt_td = market_long[market_long["date"].isin(trade_dates)].copy()

value_cols = ["ret","log_volume_change","abs_ret","vol_20d","abret_capm","abret_ff3"]
wide = mkt_td.pivot(index="date", columns="ticker", values=value_cols)
wide.columns = [f"{v}_{t}" for v, t in wide.columns]
wide = wide.reset_index().rename(columns={"date":"trade_date"})

display(wide.head())

,trade_date,ret_DKNG,ret_NKE,ret_SPY,ret_XLY,log_volume_change_DKNG,log_volume_change_NKE,log_volume_change_SPY,log_volume_change_XLY,abs_ret_DKNG,abs_ret_NKE,abs_ret_SPY,abs_ret_XLY,vol_20d_DKNG,vol_20d_NKE,vol_20d_SPY,vol_20d_XLY,abret_capm_DKNG,abret_capm_NKE,abret_capm_SPY,abret_capm_XLY,abret_ff3_DKNG,abret_ff3_NKE,abret_ff3_SPY,abret_ff3_XLY
0,2000-01-03,NaN,-0.029433,-0.009836,-0.030646,NaN,1.492121,0.945188,2.411134,NaN,0.029433,0.009836,0.030646,NaN,0.047884,0.007373,0.016392,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2000-02-01,NaN,0.025764,0.009804,0.035547,NaN,0.254996,-0.246046,2.487219,NaN,0.025764,0.009804,0.035547,NaN,0.031146,0.021565,0.021004,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2000-02-16,NaN,-0.031285,-0.014840,-0.039864,NaN,-0.281649,-0.225091,0.823509,NaN,0.031285,0.014840,0.039864,NaN,0.049541,0.015594,0.019754,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2000-03-01,NaN,-0.011051,0.007249,-0.011624,NaN,-0.240728,-0.182431,2.977672,NaN,0.011051,0.007249,0.011624,NaN,0.053370,0.013203,0.018709,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2000-04-03,NaN,0.009420,0.005801,0.029460,NaN,-0.055795,-0.083496,-2.212287,NaN,0.009420,0.005801,0.029460,NaN,0.043911,0.016677,0.023608,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [26]:
final = medals.merge(wide, on="trade_date", how="left")

# Reorder key columns
front = ["trade_date","medal_date","gold","silver","bronze","total_medals","performance_score","any_medal_dummy"]
front = [c for c in front if c in final.columns]
final = final[front + [c for c in final.columns if c not in front]]

display(final.head(10))
print("Final panel shape:", final.shape)

# Quick missing checks for core series
for col in ["ret_SPY","log_volume_change_SPY","vol_20d_SPY"]:
    if col in final.columns:
        print(col, "missing:", final[col].isna().sum())

,trade_date,medal_date,gold,silver,bronze,total_medals,performance_score,any_medal_dummy,season,ret_DKNG,ret_NKE,ret_SPY,ret_XLY,log_volume_change_DKNG,log_volume_change_NKE,log_volume_change_SPY,log_volume_change_XLY,abs_ret_DKNG,abs_ret_NKE,abs_ret_SPY,abs_ret_XLY,vol_20d_DKNG,vol_20d_NKE,vol_20d_SPY,vol_20d_XLY,abret_capm_DKNG,abret_capm_NKE,abret_capm_SPY,abret_capm_XLY,abret_ff3_DKNG,abret_ff3_NKE,abret_ff3_SPY,abret_ff3_XLY
0,2000-01-03,2000-01-01,10,6,7,23,49,1,S,NaN,-0.029433,-0.009836,-0.030646,NaN,1.492121,0.945188,2.411134,NaN,0.029433,0.009836,0.030646,NaN,0.047884,0.007373,0.016392,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2000-02-01,2000-02-01,3,5,5,13,24,1,S,NaN,0.025764,0.009804,0.035547,NaN,0.254996,-0.246046,2.487219,NaN,0.025764,0.009804,0.035547,NaN,0.031146,0.021565,0.021004,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2000-02-16,2000-02-16,0,0,2,2,2,1,S,NaN,-0.031285,-0.014840,-0.039864,NaN,-0.281649,-0.225091,0.823509,NaN,0.031285,0.014840,0.039864,NaN,0.049541,0.015594,0.019754,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2000-03-01,2000-03-01,0,0,1,1,1,1,S,NaN,-0.011051,0.007249,-0.011624,NaN,-0.240728,-0.182431,2.977672,NaN,0.011051,0.007249,0.011624,NaN,0.053370,0.013203,0.018709,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2000-04-03,2000-04-01,2,0,0,2,6,1,S,NaN,0.009420,0.005801,0.029460,NaN,-0.055795,-0.083496,-2.212287,NaN,0.009420,0.005801,0.029460,NaN,0.043911,0.016677,0.023608,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,2000-04-24,2000-04-23,0,0,1,1,1,1,S,NaN,-0.017217,-0.010924,0.010639,NaN,0.129059,0.412212,-0.150660,NaN,0.017217,0.010924,0.010639,NaN,0.029207,0.019276,0.023583,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,2000-05-10,2000-05-10,0,0,1,1,1,1,S,NaN,-0.021947,-0.022815,0.000000,NaN,-0.436589,0.605166,1.258334,NaN,0.021947,0.022815,0.000000,NaN,0.031192,0.022930,0.025838,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,2000-06-01,2000-06-01,1,0,1,2,4,1,S,NaN,0.008709,0.017354,0.007465,NaN,-0.606619,0.391817,-2.926739,NaN,0.008709,0.017354,0.007465,NaN,0.025672,0.015379,0.014972,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,2000-07-03,2000-07-01,1,0,0,1,3,1,S,NaN,0.029391,0.013673,0.013342,NaN,-0.351650,-1.641927,0.280778,NaN,0.029391,0.013673,0.013342,NaN,0.031170,0.009436,0.015076,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,2000-07-10,2000-07-09,1,0,0,1,3,1,S,NaN,-0.008955,-0.001689,0.014550,NaN,-0.276652,-0.074793,-0.359496,NaN,0.008955,0.001689,0.014550,NaN,0.027548,0.010876,0.017109,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Final panel shape: (281, 33)
ret_SPY missing: 0
log_volume_change_SPY missing: 0
vol_20d_SPY missing: 0


## 7) Export final panel CSV

This is the regression/event-study dataset.


In [27]:
OUT_PANEL = Path("../data/interim/usa/usa_medals_market_panel.csv")
final.to_csv(OUT_PANEL, index=False)
print("Wrote:", OUT_PANEL.resolve())

Wrote: /Users/ayomadeadeolu-akande/Documents/GitHub/QSS20_Final_Project/data/interim/usa/usa_medals_market_panel.csv


## 8) Optional: Export a compact regression subset

This creates a smaller CSV matching a typical regression schema.


In [28]:
keep = [
    "trade_date","medal_date","gold","silver","bronze","performance_score","any_medal_dummy",
    "ret_SPY","log_volume_change_SPY","abs_ret_SPY","vol_20d_SPY",
    "ret_NKE","abret_capm_NKE","abret_ff3_NKE",
    "ret_XLY","abret_capm_XLY","abret_ff3_XLY",
    "ret_DKNG","abret_capm_DKNG","abret_ff3_DKNG",
]
keep = [c for c in keep if c in final.columns]

reg_df = final[keep].copy()
display(reg_df.head(10))

reg_df.to_csv("../data/interim/usa/usa_medals_market_panel_regression_subset.csv", index=False)
print("Wrote: usa_medals_market_panel_regression_subset.csv")

,trade_date,medal_date,gold,silver,bronze,performance_score,any_medal_dummy,ret_SPY,log_volume_change_SPY,abs_ret_SPY,vol_20d_SPY,ret_NKE,abret_capm_NKE,abret_ff3_NKE,ret_XLY,abret_capm_XLY,abret_ff3_XLY,ret_DKNG,abret_capm_DKNG,abret_ff3_DKNG
0,2000-01-03,2000-01-01,10,6,7,49,1,-0.009836,0.945188,0.009836,0.007373,-0.029433,NaN,NaN,-0.030646,NaN,NaN,NaN,NaN,NaN
1,2000-02-01,2000-02-01,3,5,5,24,1,0.009804,-0.246046,0.009804,0.021565,0.025764,NaN,NaN,0.035547,NaN,NaN,NaN,NaN,NaN
2,2000-02-16,2000-02-16,0,0,2,2,1,-0.014840,-0.225091,0.014840,0.015594,-0.031285,NaN,NaN,-0.039864,NaN,NaN,NaN,NaN,NaN
3,2000-03-01,2000-03-01,0,0,1,1,1,0.007249,-0.182431,0.007249,0.013203,-0.011051,NaN,NaN,-0.011624,NaN,NaN,NaN,NaN,NaN
4,2000-04-03,2000-04-01,2,0,0,6,1,0.005801,-0.083496,0.005801,0.016677,0.009420,NaN,NaN,0.029460,NaN,NaN,NaN,NaN,NaN
5,2000-04-24,2000-04-23,0,0,1,1,1,-0.010924,0.412212,0.010924,0.019276,-0.017217,NaN,NaN,0.010639,NaN,NaN,NaN,NaN,NaN
6,2000-05-10,2000-05-10,0,0,1,1,1,-0.022815,0.605166,0.022815,0.022930,-0.021947,NaN,NaN,0.000000,NaN,NaN,NaN,NaN,NaN
7,2000-06-01,2000-06-01,1,0,1,4,1,0.017354,0.391817,0.017354,0.015379,0.008709,NaN,NaN,0.007465,NaN,NaN,NaN,NaN,NaN
8,2000-07-03,2000-07-01,1,0,0,3,1,0.013673,-1.641927,0.013673,0.009436,0.029391,NaN,NaN,0.013342,NaN,NaN,NaN,NaN,NaN
9,2000-07-10,2000-07-09,1,0,0,3,1,-0.001689,-0.074793,0.001689,0.010876,-0.008955,NaN,NaN,0.014550,NaN,NaN,NaN,NaN,NaN


Wrote: usa_medals_market_panel_regression_subset.csv
